# Classificazione binaria



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Le reti possono risolvere anche problemi di **classificazione**. Gran parte di quanto hai studiato resta valido; cambiano soprattutto loss e uscita finale.

# Classificazione binaria
Acquisto sì/no, frode sì/no, presenza o assenza di un segnale sono esempi di classificazione in due classi. Le etichette testuali, come `Yes`/`No` o `Dog`/`Cat`, vengono codificate con **0** e **1**.

# Accuracy e cross-entropy
L'**accuracy** è la frazione di previsioni corrette:

```text
accuracy = number_correct / total
```

Vale 1 se tutte le risposte sono corrette. È particolarmente interpretabile quando le classi hanno frequenze simili. Cambia però a salti quando cambia una decisione; SGD richiede una loss sulla quale calcolare gradienti utili.

Usiamo quindi la **cross-entropy**, che confronta le probabilità previste con le etichette reali. Il testo originale la presenta come una distanza fra distribuzioni: è un'intuizione, non una distanza matematica simmetrica.

![Accuracy e cross-entropy](https://storage.googleapis.com/kaggle-media/learn/images/DwVV9bR.png)

Vorremmo assegnare probabilità alta alla classe corretta. Più bassa è tale probabilità, maggiore è la penalità. Per la classificazione useremo cross-entropy come loss e potremo monitorare anche accuracy. Migliorare la loss non garantisce a ogni epoca un aumento dell'accuracy.

# Probabilità con la sigmoid
Per trasformare un'uscita reale in un valore fra 0 e 1 aggiungiamo l'attivazione **sigmoid**.

![Funzione sigmoid](https://storage.googleapis.com/kaggle-media/learn/images/FYbRvJo.png)

Una **soglia** trasforma la probabilità in una classe: nel caso usuale 0,5, valori inferiori indicano classe 0 e quelli superiori classe 1. La [BinaryAccuracy di Keras](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/BinaryAccuracy) usa una soglia configurabile, predefinita a 0,5; il comportamento esatto sul punto di soglia dipende dalla convenzione usata.

# Esempio: ionosfera
Il dataset [Ionosphere](https://archive.ics.uci.edu/ml/datasets/Ionosphere) descrive segnali radar diretti verso l'ionosfera. Il compito è distinguere segnali che mostrano una struttura da quelli che non la mostrano.


In [ ]:

import pandas as pd
from IPython.display import display

ion = pd.read_csv('../input/dl-course-data/ion.csv', index_col=0)
display(ion.head())

df = ion.copy()
df['Class'] = df['Class'].map({'good': 0, 'bad': 1})

df_train = df.sample(frac=0.7, random_state=0)
df_valid = df.drop(df_train.index)

max_ = df_train.max(axis=0)
min_ = df_train.min(axis=0)

df_train = (df_train - min_) / (max_ - min_)
df_valid = (df_valid - min_) / (max_ - min_)
df_train.dropna(axis=1, inplace=True) # Rimuovi la feature vuota nella colonna 2.
df_valid.dropna(axis=1, inplace=True)

X_train = df_train.drop('Class', axis=1)
X_valid = df_valid.drop('Class', axis=1)
y_train = df_train['Class']
y_valid = df_valid['Class']

Definiamo il modello come nella regressione, ma con **`activation='sigmoid'`** nell'ultimo layer, per ottenere una probabilità.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(4, activation='relu', input_shape=[33]),
    layers.Dense(4, activation='relu'),    
    layers.Dense(1, activation='sigmoid'),
])

Con **`compile`** aggiungi **`binary_crossentropy`**, **`binary_accuracy`** e l'ottimizzatore **Adam**. Le versioni binarie sono adatte a due classi; il caso multiclasse richiede altre scelte.


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

Il problema può richiedere molte epoche: aggiungiamo una callback di early stopping.


In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta=0.001,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=1000,
    callbacks=[early_stopping],
    verbose=0, # Nascondi l'output perché le epoche sono numerose.
)

Esaminiamo le curve e i migliori valori di loss e accuracy sul validation set. Ricorda che l'early stopping ripristina i pesi migliori rispetto alla misura monitorata.


In [ ]:
history_df = pd.DataFrame(history.history)
# Inizia il grafico dall'epoca 5.
history_df.loc[5:, ['loss', 'val_loss']].plot()
history_df.loc[5:, ['binary_accuracy', 'val_binary_accuracy']].plot()

print(("Best Validation Loss: {:0.4f}" +\
      "\nBest Validation Accuracy: {:0.4f}")\
      .format(history_df['val_loss'].min(), 
              history_df['val_binary_accuracy'].max()))

Ora **[prevedi le cancellazioni alberghiere](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex6.html)** con il dataset *Hotel Cancellations*.
